# フォルニー（Forney）法によるエラー値の算出と最終修復

<a href="https://colab.research.google.com/github/Tsukumo-999/qr-code-from-scratch/blob/master/Step3/4_forney_algorithm.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

第11回記事で行った、判明したエラーの場所からエラーの「値」を算出し、データを完全に修復するフォルニー法の実験ノートブックです。

### 1. これまでの計算結果の引き継ぎ
これまでのノートブックで導き出された「受信データ」「シンドローム」「エラー位置多項式」「エラーの位置（インデックス）」をセットアップします。

In [7]:
# ガロア体 GF(2^8) の準備
exp_table = [0] * 256
log_table = [0] * 256
v = 1
for i in range(255):
    exp_table[i] = v
    log_table[v] = i
    v <<= 1
    if v & 0x100: v ^= 0x11D
exp_table[255] = exp_table[0]

def gf_mul(x, y):
    if x == 0 or y == 0: return 0
    return exp_table[(log_table[x] + log_table[y]) % 255]

def gf_div(x, y):
    if y == 0: raise ZeroDivisionError("0除算")
    if x == 0: return 0
    return exp_table[(log_table[x] - log_table[y]) % 255]

# 意図的に2箇所（インデックス1と3）を壊した受信メッセージ
received_message = [72, 99, 76, 200, 162, 112, 246]
msg_length = len(received_message)
num_ec_codewords = 4
 
# 第8回のシンドローム計算結果
syndromes = [139, 21, 219, 80]

# 第9回のBM法で得られたエラー位置多項式 σ(x)
C = [1, 40, 29]

# 第10回のチェン探索で発見したエラーの場所 (i は後ろからの位置、idx は配列のインデックス)
error_positions = [5, 3] # i = 5 (idx=1), i = 3 (idx=3)

print("▼ 引き継いだデータ ▼")
print(f"受信データ: {received_message}")
print(f"シンドローム: {syndromes}")
print(f"エラー位置多項式: {C}")
print(f"エラーの場所(i): {error_positions}")

▼ 引き継いだデータ ▼
受信データ: [72, 99, 76, 200, 162, 112, 246]
シンドローム: [139, 21, 219, 80]
エラー位置多項式: [1, 40, 29]
エラーの場所(i): [5, 3]


### 2. エラー評価多項式 Ω(x) の計算
シンドローム多項式 $S(x)$ とエラー位置多項式 $C(x)$ を掛け合わせて、エラーの「大きさ」情報を持つ多項式 $\Omega(x)$ を作成します。

In [8]:
# Ω(x) = S(x) * C(x) mod x^4
Omega = [0] * num_ec_codewords

for i in range(num_ec_codewords):
    for j in range(len(C)):
        if i - j >= 0:
            Omega[i] ^= gf_mul(C[j], syndromes[i - j])
            
print("▼ エラー評価多項式 Ω(x) ▼")
print(f"Omega = {Omega}")

▼ エラー評価多項式 Ω(x) ▼
Omega = [139, 137, 0, 0]


### 3. フォルニー法によるエラー値の計算とデータ修復
チェン探索で見つけた各エラー位置に対して、$\Omega(x)$ と $C(x)$の微分（奇数項のみ）を使ってエラー値（反転すべき値）を計算します。

In [9]:
print("【フォルニー法によるエラー修復開始】\n")
repaired_message = received_message.copy()

for pos in error_positions:
    # 代入する x_inv = α^{-pos}
    x_inv = 1 if pos == 0 else exp_table[255 - pos]
    
    # 1. Ω(x_inv) の計算
    omega_val = 0
    x_pow = 1
    for coef in Omega:
        omega_val ^= gf_mul(coef, x_pow)
        x_pow = gf_mul(x_pow, x_inv)
        
    # 2. C'(x_inv) (微分) の計算
    # ガロア体の微分は「偶数乗の項が消滅し、奇数乗の項が残る」
    deriv_val = 0
    x_pow = 1
    for j in range(1, len(C), 2):
        deriv_val ^= gf_mul(C[j], x_pow)
        x_pow = gf_mul(x_pow, gf_mul(x_inv, x_inv)) # 微分後は x^0, x^2, x^4... と増える
        
    # 3. エラー値の算出
    # QRコードの公式 e = (Ω(x_inv) / C'(x_inv)) * α^pos
    error_value = gf_mul(gf_div(omega_val, deriv_val), exp_table[pos])
    
    idx = (msg_length - 1) - pos
    print(f"  [修復] インデックス {idx} のエラー値は '{error_value}' と判明！")
    
    # 4. XOR演算でデータを反転させて修復
    repaired_message[idx] ^= error_value

print("\n====================================")
print(f" 受信データ: {received_message}")
print(f" 修復データ: {repaired_message}")
print("====================================")

# 文字列に変換して最終確認
decoded_chars = [chr(c) for c in repaired_message[:3]]
print(f"\n 復元された文字列: {''.join(decoded_chars)} ")

【フォルニー法によるエラー修復開始】

  [修復] インデックス 1 のエラー値は '34' と判明！
  [修復] インデックス 3 のエラー値は '169' と判明！

 受信データ: [72, 99, 76, 200, 162, 112, 246]
 修復データ: [72, 65, 76, 97, 162, 112, 246]

 復元された文字列: HAL 
